# 04 - Evaluation

## Setup and Dependencies

In [ ]:
# Install Dependencies
!pip install -q -U "weaviate-client<4.0.0" pypdf rouge-score
!pip install -q -U langchain langchain-community langchain-text-splitters langchain-google-genai sentence-transformers python-dotenv
!pip install -q -U "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q -U --no-deps "trl<0.9.0" peft accelerate bitsandbytes

import os
repo_name = "zuucrew-ai-aee-mini-project-1"
repo_url = f"https://github.com/Sulamaxx/{repo_name}.git"

if 'google.colab' in str(get_ipython()):
    if not os.path.exists(f"/content/{repo_name}"):
        print(f"Cloning {repo_name}...")
        !git clone {repo_url}
    
    os.chdir(f"/content/{repo_name}")
    print(f"Switched to repo directory: {os.getcwd()}")
    
    try:
        from google.colab import userdata
        for key in ["GEMINI_API_KEY", "GOOGLE_API_KEY"]:
            val = userdata.get(key)
            if val:
                os.environ[key] = val
                print(f"{key} loaded from Colab Secrets.")
    except Exception: pass

import warnings
warnings.filterwarnings("ignore")

## Initialize Models

In [ ]:
import torch
import time
import os
from unsloth import FastLanguageModel
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from src.services.llm_services import load_config, get_llm, get_text_embeddings

load_dotenv()
config = load_config("src/config/config.yaml")

# RAG Setup
llm = get_llm(config)
embeddings = get_text_embeddings(config)

import weaviate
from weaviate.embedded import EmbeddedOptions
client = weaviate.Client(
    embedded_options=EmbeddedOptions(),
    additional_headers={"X-Goog-Api-Key": os.environ.get("GEMINI_API_KEY")}
)

# Finetuned Setup
model_path = "artifacts/intern_adapters"

if not os.path.exists(model_path) and 'google.colab' in str(get_ipython()):
    drive_path = "/content/drive/MyDrive/AIEngineering/Project1/intern_adapters"
    print(f"Adapters not found in repo. Checking Google Drive at {drive_path}...")
    
    from google.colab import drive
    drive.mount('/content/drive')
    
    if os.path.exists(drive_path):
        os.makedirs(model_path, exist_ok=True)
        !cp -r {drive_path}/* {model_path}/
        print("Adapters loaded from Google Drive.")

if not os.path.exists(model_path):
    print(f"ERROR: Adapters not found at {model_path}")
else:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = model_path,
        load_in_4bit = True,
    )
    FastLanguageModel.for_inference(model)
    print("All models initialized.")

## Ingest and Index (for RAG)

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader("data/pdfs/2024AnnualReport.pdf")
docs = loader.load()

# Chunking
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = text_splitter.split_documents(docs)

# Indexing into Weaviate
class_obj = {
    "class": "LibrarianChunk",
    "vectorizer": "none",
    "properties": [
        {"name": "text", "dataType": ["text"]},
        {"name": "page_number", "dataType": ["int"]}
    ]
}

if client.schema.exists("LibrarianChunk"): 
    client.schema.delete_class("LibrarianChunk")
client.schema.create_class(class_obj)

with client.batch as batch:
    batch.batch_size = 100
    for i, d in enumerate(chunks):
        properties = {
            "text": d.page_content,
            "page_number": d.metadata.get("page", 0) + 1
        }
        batch.add_data_object(properties, "LibrarianChunk", vector=embeddings.embed_query(d.page_content))

print(f"{len(chunks)} chunks indexed for The Librarian.")

## Inference Wrappers

In [ ]:
def query_intern(question):
    """Fine-tuned Llama-3-2-3B inference."""
    prompt = f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
    inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")
    
    start_time = time.perf_counter()
    outputs = model.generate(
        **inputs, 
        max_new_tokens = 256, 
        use_cache = True,
        repetition_penalty = 1.2 
    )
    end_time = time.perf_counter()
    
    response = tokenizer.batch_decode(outputs)
    answer = response[0].split("assistant<|end_header_id|>\n\n")[-1].replace("<|eot_id|>", "").strip()
    return answer, (end_time - start_time) * 1000  # ms

from sentence_transformers import CrossEncoder
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def hybrid_search(query, limit=10):
    query_vec = embeddings.embed_query(query)
    results = (client.query
               .get("LibrarianChunk", ["text", "page_number"])
               .with_hybrid(query=query, vector=query_vec, alpha=0.5)
               .with_limit(limit)
               .do())
    return results["data"]["Get"]["LibrarianChunk"]

def query_librarian(question):
    """RAG pipeline: Hybrid Search -> Rerank -> Final Generation."""
    start_time = time.perf_counter()
    
    # Retrieval (Hybrid)
    initial_results = hybrid_search(question, limit=10)
    if not initial_results: 
        return "No info found.", (time.perf_counter() - start_time) * 1000
    
    # Reranking
    passages = [res["text"] for res in initial_results]
    scores = reranker.predict([[question, p] for p in passages])
    ranked = sorted(zip(initial_results, scores), key=lambda x: x[1], reverse=True)
    
    best_context = ranked[0][0]["text"]
    page = ranked[0][0]["page_number"]
    
    # Generation
    prompt = f"""You are a precise librarian auditing Uber. Answer based ONLY on context.
    
    Context (Source Page {page}):
    {best_context}
    
    Question: {question}
    
    Answer:"""
    
    response = llm.invoke(prompt)
    answer = response.content if hasattr(response, 'content') else str(response)
    
    end_time = time.perf_counter()
    return answer, (end_time - start_time) * 1000

## Evaluations Metrics

In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def get_rouge_l(ground_truth, generated):
    scores = scorer.score(ground_truth, generated)
    return scores['rougeL'].fmeasure

def score_answer(question, ground_truth, answer):
    """LLM-as-a-Judge using Gemini."""
    prompt = f"""You are an expert financial auditor. Rate the generated answer based on the ground truth.
    
    Question: {question}
    Ground Truth: {ground_truth}
    Generated Answer: {answer}
    
    Provide TWO scores from 1 to 5 (5 is best):
    1. Faithfulness: Is the answer strictly derived from facts?
    2. Accuracy: Does it correctly answer the question?
    
    Respond ONLY with a JSON object: {{"faithfulness": X, "accuracy": Y}}"""
    
    try:
        response = llm.invoke(prompt)
        score_text = response.content if hasattr(response, 'content') else str(response)
        # Simple cleaning if LLM adds markdown blocks
        score_text = score_text.replace("```json", "").replace("```", "").strip()
        import json as py_json
        return py_json.loads(score_text)
    except Exception as e:
        print(f"Judge error: {e}")
        return {"faithfulness": 0, "accuracy": 0}

In [ ]:
import pandas as pd
import json 
from tqdm.auto import tqdm

test_file = "artifacts/data_factory/golden_test_set.jsonl"

# Load the data
with open(test_file, "r") as f:
    test_data = [json.loads(line) for line in f]

subset = test_data[:10]
results = []

print(f"Starting Evaluation on {len(subset)} questions...")

for item in tqdm(subset):
    q = item["instruction"]
    gt = item["output"]
    
    # Query Intern
    ans_i, lat_i = query_intern(q)
    rouge_i = get_rouge_l(gt, ans_i)
    judgement_i = score_answer(q, gt, ans_i)
    
    # Query Librarian
    ans_l, lat_l = query_librarian(q)
    rouge_l = get_rouge_l(gt, ans_l)
    judgement_l = score_answer(q, gt, ans_l)
    
    results.append({
        "Question": q,
        "Intern_ROUGE": rouge_i,
        "Intern_Latency": lat_i,
        "Intern_Faithfulness": judgement_i["faithfulness"],
        "Intern_Accuracy": judgement_i["accuracy"],
        "Librarian_ROUGE": rouge_l,
        "Librarian_Latency": lat_l,
        "Librarian_Faithfulness": judgement_l["faithfulness"],
        "Librarian_Accuracy": judgement_l["accuracy"],
    })

df = pd.DataFrame(results)
print("Evaluation Complete.")

## Summary Report

In [ ]:
summary = pd.DataFrame({
    "Metric": ["Avg ROUGE-L", "Avg Latency (ms)", "Avg Faithfulness", "Avg Accuracy"],
    "The Intern (Fine-Tuned)": [
        df["Intern_ROUGE"].mean(),
        df["Intern_Latency"].mean(),
        df["Intern_Faithfulness"].mean(),
        df["Intern_Accuracy"].mean()
    ],
    "The Librarian (RAG)": [
        df["Librarian_ROUGE"].mean(),
        df["Librarian_Latency"].mean(),
        df["Librarian_Faithfulness"].mean(),
        df["Librarian_Accuracy"].mean()
    ]
})

display(summary)

best_acc = "The Librarian" if summary.iloc[3, 2] > summary.iloc[3, 1] else "The Intern"
print(f"Higher Accuracy Provider: {best_acc}")